Load Configuration

In [0]:
%run ./00_0_config

Data Files Generation

In [0]:
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime, timedelta
import random
from pyspark.sql import Row

# ============================================================
# CONFIGURAION — min/max values per column
# ============================================================

column_config = [
    # (naziv_kolone,  min_norm, max_norm, min_outlier, max_outlier, type, has_outlier)
    #(date_time,      0,     23,        0,     2,     "date"),
    ("cell_id",             1,     50,        1,     50,    "int", False),
    ("subscriber_id",       1,   1000,        1,     1000,  "int", False),
    ("measurement_type",    0,     1,        0,     1,      "int", False),
    ("connection_type",     0,     1,        0,     1,      "int", False),
    ("call_direction",      0,     1,        0,     1,      "int", False),
    ("device_model",        0,     1,        0,     1,      "int", False),
    ("signal_strength",     -100,  -40,     -120,  -30,     "int", True),
    ("rsrp",                -120, -80,      -60,    -50,    "int", True),
    ("rsrq",                -120, -80,      -60,    -50,    "int", True),
    ("sinr",                -120, -80,      -60,    -50,    "int", True),
    ("ecio",                -120, -80,      -60,    -50,    "int", True),
    ("mos",                 1.0,    5.0,    10.0,   15.0,   "float", True),
    ("jitter_ms",           1.0,   50.0,   100.0, 500.0,    "float", True),
    ("cqi",                 6,     15,        0,     2,     "int", True),
    ("mos",                 1.0,    2.0,    5.0,    6.0,    "float", True),
    ("cqi",                 6,     15,        0,     2,     "int", True),
    ("mos",                 1.0,    5.0,     0.1,   0.9,    "float", True),
    ("jitter_ms",           1.0,   50.0,   100.0, 500.0,    "float", True),
    ("latency_ms",          5.0,  100.0,   300.0, 999.0,    "float", True),
    ("packet_loss_pct",     0.0,    2.0,    10.0,  50.0,    "float", True),
    ("download_throughput", 1.0, 100.0,    0.1,   0.5,      "float", True),
    ("upload_throughput",   0.5,  50.0,    0.1,   0.4,      "float", True),
    ("rsrp_dbm",            -80.0,  -44.0,  -140.0,-110.0,  "float", True),
    ("rsrq_db",             -10.0,   -3.0,   -20.0, -15.0,  "float", True),
    ("sinr_db",             0.0,   30.0,   -10.0,  -1.0,    "float", True),
    ("throughput_dl_mbps",  1.0, 100.0,    0.1,   0.5,      "float", True),
    ("throughput_ul_mbps",  0.5,  50.0,    0.1,   0.4,      "float", True),
    ("cqi",                 6,     15,        0,     2,     "int", True),
]

TECHNOLOGIES = ["voice", "txt", "3g", "4g", "5g", "vod"]
OUTLIER_PROBABILITY = 0.02  # 2% outliera

# ============================================================
# GENERATOR
# ============================================================

def generate_value(min_norm, max_norm, min_outlier, max_outlier, type, is_outlier, has_outlier):
    if is_outlier and has_outlier:
        val = random.uniform(min_outlier, max_outlier)
    else:
        val = random.uniform(min_norm, max_norm)
    
    return int(round(val)) if type == "int" else round(val, 2)


def generate_row(row_id, base_time, config, outlier_prob):
    is_outlier = random.random() < outlier_prob
    
    row = {
        "row_id":      row_id,
        "timestamp":   (base_time + timedelta(seconds=row_id)).isoformat(),
        "technology":  random.choice(TECHNOLOGIES),
        "cell_id":     f"CELL_{random.randint(1, 50):03d}",
        "ue_id":       f"UE_{random.randint(1, 1000):04d}",
        "latitude":    round(random.uniform(44.0, 46.0), 6),
        "longitude":   round(random.uniform(19.0, 22.0), 6),
        "is_outlier":  is_outlier,  # ground truth za validaciju autoencodera kasnije
    }
    
    for col_name, min_norm, max_norm, min_out, max_out, type, has_outlier in config:
        row[col_name] = generate_value(
            min_norm, max_norm, min_out, max_out, type, is_outlier, has_outlier
        )
    
    return row


def generate_dataset(output_dir, output_type, n_rows=1_000_000, batch_size=10_000):
    os.makedirs(output_dir, exist_ok=True)
    base_time = datetime.now() #- timedelta(days=2)
    total_outliers = 0
    all_rows = []
    
    for batch_num in range(0, n_rows, batch_size):
        batch = []
        for i in range(batch_num, min(batch_num + batch_size, n_rows)):
            row = generate_row(i, base_time, column_config, OUTLIER_PROBABILITY)

            row["processing_date"] = base_time.strftime("%Y-%m-%d")

            if row["is_outlier"]:
                total_outliers += 1
            all_rows.append(row)
        
        #filename = f"{output_dir}/batch_{batch_num:08d}.json"
        filename = f"{output_dir}/date={base_time.strftime('%Y-%m-%d')}"
        if output_type == "local":    
            with open(filename, "w") as f:
                for row in batch:
                    f.write(json.dumps(row) + "\n")  # newline-delimited JSON
        
        if output_type == "azure":
            spark_df = spark.createDataFrame(all_rows)
            spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
            spark_df.repartition(100).write.mode("overwrite").partitionBy("processing_date").json(output_dir)
    
    print(f"\nGotovo! Ukupno redova: {n_rows}, outliera: {total_outliers}")


# ============================================================
# POKRETANJE
# ============================================================

generate_dataset(
    n_rows=1_000_00,
    batch_size=10_000,
    #output_type="local",
    #output_dir="./data/raw"
    output_type="azure",
    #output_dir="abfss://data-lake@storagetelcopoc2026.dfs.core.windows.net/landing/"
    output_dir=paths["raw"]
)

Checks

In [0]:
#spark.catalog.refreshByPath(paths["raw"])

df = spark.read.json("abfss://data-lake@storagetelcopoc2026.dfs.core.windows.net/00-raw/")

display(df)

In [0]:
df.printSchema()

In [0]:
df.groupBy("technology").count().orderBy("count", ascending=False).display()

In [0]:
df.groupBy("is_outlier").count().display()

In [0]:
display(df.count())

In [0]:
display(df.select("processing_date").distinct().orderBy("processing_date"))

df = spark.read.json(paths["raw"])
display(df.groupBy("processing_date").count().orderBy("processing_date"))

In [0]:
%sql
show external locations